# Network Intrusion Detection: Exploratory Data Analysis (EDA)
**ML Capstone | Person 1: Data & EDA Lead**
**Dataset:** CICIDS2017 (`GeneratedLabelledFlows`, the version that contains `Timestamp` and IP addresses)

---

### What this notebook does
1. Explains **what the data is** and **why we chose it**.
2. Loads and **inspects** the data (size, types, labels, class imbalance).
3. Checks **data quality** (missing / infinite values, duplicates, constant columns) and **cleans** it.
4. Explores the data by answering **10 EDA questions** (distributions, benign vs attack, time, IPs).
5. Runs a **correlation analysis** and decides which features to keep.
6. Saves a **clean dataset** and a **feature list** for the rest of the team (Person 2 builds the models).

### How to run
Put the CSV files in `data/raw/` and run the cells from top to bottom. If your computer is slow or has little RAM, set `SAMPLE_FRAC = 0.2` in the first code cell (it keeps a random 20% of rows **in time order**).

> Every section follows the same pattern: **question -> why we ask it -> code -> what the result means**.

## 1. About the data and why we use it

### What is CICIDS2017?
CICIDS2017 was created by the **Canadian Institute for Cybersecurity (University of New Brunswick)**. They built a small lab network, simulated the behaviour of 25 normal users (web, FTP, SSH, email), and then launched real attacks against it during **5 days (Monday 3 July to Friday 7 July 2017)**. All traffic was recorded, then converted into **flows** by a tool called *CICFlowMeter*, which computes more than 80 statistics for each flow. Every flow is **labelled** as `BENIGN` (normal) or with the name of the attack.

### What is a "flow"?
A **flow** is a group of packets that belong to the same conversation (same source IP, destination IP, ports and protocol). Instead of looking at every single packet, we look at one row per conversation with summary numbers: how long it lasted, how many packets and bytes went each way, how fast packets arrived, which TCP flags were used, and so on.

> **One row = one network flow.** `Fwd` (forward) means from the client to the server, `Bwd` (backward) means from the server to the client.

### What attacks happen on which day (typical layout, we verify it in Section 5)
| Day | Content |
|---|---|
| Monday | Benign traffic only |
| Tuesday | FTP-Patator and SSH-Patator (brute force) |
| Wednesday | DoS attacks (Hulk, GoldenEye, slowloris, Slowhttptest) and Heartbleed |
| Thursday | Web attacks (Brute Force, XSS, SQL Injection) and Infiltration |
| Friday | Botnet, PortScan and DDoS |

### Why this dataset fits our project
| Reason | Why it matters for us |
|---|---|
| It has a real **`Timestamp`** | Our project is **time-series** anomaly detection. We need the flows in time order to build sliding windows. |
| It has **Source IP** | We can check whether the same machine sends many flows in a row (sequential behaviour). |
| **Realistic benign traffic** and **many attack types** | We can train on normal traffic and test on several kinds of attack. |
| **Labelled** flows | We can measure precision / recall / F1 / ROC-AUC. |
| **Very widely used** benchmark | Our results can be compared with published work. |

### Known limitations (we mention these in the report)
* The data is from **2017** and from a **lab network**, so real networks may look different.
* It is **highly imbalanced**: most flows are benign.
* Published studies reported **labelling and flow-construction problems** in the original CSV files, so a small amount of label noise is possible.
* A model that learns these attacks may **not generalise** to new (zero-day) attacks.

## 2. Setup and configuration
Here we import the libraries, set the paths, and define two options:
* `SAMPLE_FRAC`: use `1.0` for all data, or a smaller number (for example `0.2`) on a weak computer.
* `SEED`: a fixed random seed so that the results are **reproducible**.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)
plt.rcParams["figure.dpi"] = 100

# ---------------- Configuration ----------------
SEED = 42
SAMPLE_FRAC = 1.0          # 1.0 = all rows; 0.2 = random 20% of rows (kept in time order)
PALETTE = {"Benign": "#4C72B0", "Attack": "#C44E52"}

# Find the folder that holds the CSV files (works from notebooks/ or from the project root)
CANDIDATES = [Path("../data/raw"), Path("data/raw"), Path(".")]
DATA_DIR = next((p for p in CANDIDATES if p.exists() and any(p.glob("*.csv"))), None)
assert DATA_DIR is not None, "No CSV files found. Put the CICIDS2017 CSV files in data/raw/"

OUT_DIR = Path("../data/processed") if Path("../data").exists() else Path("data/processed")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name):
    """Save the current figure (useful later for the slides)."""
    plt.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)

print("Data folder  :", DATA_DIR.resolve())
print("Output folder:", OUT_DIR.resolve())

## 3. Load the data
**What we do:** read every CSV file, join them into one table, and add a column `source_file` so we always know which file (day) a row came from.

**Why `encoding="latin1"`:** these files contain special characters (for example in the *Web Attack* labels) that make the default UTF-8 reader fail.

**Why we clean the column names:** in this dataset many column names start with a space (for example `" Label"`). We remove those spaces so that `df["Label"]` works.

**Why we sometimes see `.1` columns:** a few files contain a duplicated column (`Fwd Header Length.1`). It is an exact copy, so we drop it.

In [ ]:
ALIASES = {"Src IP": "Source IP", "Dst IP": "Destination IP",
           "Src Port": "Source Port", "Dst Port": "Destination Port"}

def normalise_columns(d):
    d.columns = d.columns.str.strip()                 # " Label" -> "Label"
    d = d.rename(columns=ALIASES)
    dup = [c for c in d.columns if c.endswith(".1") and c[:-2] in d.columns]
    return d.drop(columns=dup)                        # remove exact duplicate columns

def load_csv(path):
    kwargs = dict(encoding="latin1", low_memory=False)
    if SAMPLE_FRAC < 1:                               # random subsample, order is preserved
        rng = np.random.default_rng(SEED)
        kwargs["skiprows"] = lambda i: i > 0 and rng.random() > SAMPLE_FRAC
    d = normalise_columns(pd.read_csv(path, **kwargs))
    d["source_file"] = path.stem
    return d

files = sorted(DATA_DIR.glob("*.csv"))
print(f"Found {len(files)} CSV file(s)\n")
frames = []
for f in files:
    part = load_csv(f)
    print(f"{f.name:<62}{part.shape[0]:>10,} rows x {part.shape[1]} columns")
    frames.append(part)

df = pd.concat(frames, ignore_index=True)
del frames
print(f"\nTotal: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 4. Understand the columns
We split the columns into three groups. This split is important because **only the third group is used by the model**.

| Group | Columns | Used as model input? |
|---|---|---|
| **Identifiers** | `Flow ID`, `Source IP`, `Source Port`, `Destination IP`, `Destination Port`, `Timestamp` | No. IPs/ports would let a model memorise the lab machines instead of learning behaviour. We use `Timestamp` only to **order** the data, and `Source IP` only for **EDA**. |
| **Target** | `Label` (attack name) and `is_attack` (0 = benign, 1 = attack, which we create) | No. It is the answer we want to detect. |
| **Features** | about 78 numeric columns computed by CICFlowMeter | **Yes** |

**The feature families (so you can explain them):**
* **Duration and rate:** `Flow Duration`, `Flow Bytes/s`, `Flow Packets/s`.
* **Packet counts and sizes:** `Total Fwd/Backward Packets`, `Fwd/Bwd Packet Length Max/Min/Mean/Std`, `Packet Length ...`.
* **Inter-arrival times (IAT):** the time gap between packets (`Flow IAT ...`, `Fwd IAT ...`, `Bwd IAT ...`). Bots and DoS tools send packets with very regular or very fast timing.
* **TCP flags:** counts of `SYN`, `ACK`, `FIN`, `RST`, `PSH`, `URG` flags. A port scan or SYN flood has a very unusual flag pattern.
* **Header, bulk and subflow statistics**, **window sizes**, and **active / idle times**.

**Data-type problem we must handle:** some columns (such as `Flow Bytes/s`) contain the text `Infinity` or `NaN`, so pandas reads them as text. We convert them to numbers and **count the infinite values first** so we can report them.

In [ ]:
ID_COLS = [c for c in ["Flow ID", "Source IP", "Source Port", "Destination IP",
                       "Destination Port", "Timestamp"] if c in df.columns]
META_COLS = ID_COLS + ["Label", "source_file"]
feat = [c for c in df.columns if c not in META_COLS]

# --- Clean the Label text (a special dash character appears in the Web Attack names)
df["Label"] = (df["Label"].astype(str)
                 .str.replace(r"[^\x20-\x7E]", "-", regex=True)
                 .str.replace(r"\s+", " ", regex=True).str.strip())

# --- Target: 0 = benign, 1 = any attack
df["is_attack"] = (df["Label"].str.upper() != "BENIGN").astype("int8")

# --- Make every feature numeric (text such as 'Infinity' -> inf, 'NaN' -> NaN)
converted = []
for c in feat:
    if not pd.api.types.is_numeric_dtype(df[c]):
        s = (df[c].astype("string").str.strip().str.lower()
               .replace({"infinity": "inf", "-infinity": "-inf"}))
        df[c] = pd.to_numeric(s, errors="coerce").astype("float64")
        converted.append(c)

print("Identifier columns :", ID_COLS)
print(f"Feature columns    : {len(feat)}")
print("Converted from text:", converted if converted else "none")

---
# Part A: Data understanding
## Q1. How big is the data, and what does it look like?
**Why we ask:** the size decides whether we can load everything in memory, and the column types tell us what cleaning is needed. The table of *rows per file and label* also **verifies which attacks are in which day**, which matters later when we split the data in time.

In [ ]:
mem_gb = df.memory_usage(deep=True).sum() / 1e9
print(f"Rows: {len(df):,} | Columns: {df.shape[1]} | Memory: {mem_gb:.2f} GB")
print("\nColumn types:")
print(df.dtypes.astype(str).value_counts().to_string())

print("\nRows per file and label (this shows which attack is in which day):")
display(pd.crosstab(df["source_file"], df["Label"]).T)

print("\nFirst rows:")
display(df.head())

**What it means:** we have millions of flows described by around 78 numeric features. The crosstab confirms that attacks are **concentrated in specific files (days)** instead of being spread evenly. Monday contains benign traffic only, which is useful for training an anomaly detector on "normal" behaviour.

## Q2. Which attack types exist, and how many flows does each have?
**Why we ask:** we need to know **what we are trying to detect**. Some attacks have millions of flows (DoS, DDoS, PortScan) while others have only a handful (Heartbleed, SQL Injection), so some will be very hard to learn.

## Q3. How imbalanced is the data (benign vs attack)?
**Why we ask:** in intrusion detection normal traffic is far more common than attacks. When 80% of the data is benign, a model that always says "benign" would score **80% accuracy without detecting anything**. This is why we will report **precision, recall, F1 and ROC-AUC** instead of accuracy.

In [ ]:
counts = df["Label"].value_counts()
label_table = pd.DataFrame({"flows": counts, "percent": (counts / len(df) * 100).round(3)})
display(label_table)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]})

colors = [PALETTE["Benign"] if l.upper() == "BENIGN" else PALETTE["Attack"] for l in counts.index]
axes[0].barh(counts.index[::-1], counts.values[::-1], color=colors[::-1])
axes[0].set_xscale("log")
axes[0].set_xlabel("Number of flows (log scale)")
axes[0].set_title("Q2. Flows per label (log scale so rare attacks are visible)")

binary = df["is_attack"].map({0: "Benign", 1: "Attack"}).value_counts()
axes[1].bar(binary.index, binary.values, color=[PALETTE[k] for k in binary.index])
for i, v in enumerate(binary.values):
    axes[1].text(i, v, f"{v / len(df) * 100:.1f}%", ha="center", va="bottom", fontweight="bold")
axes[1].set_title("Q3. Benign vs attack")
axes[1].set_ylabel("Number of flows")
plt.tight_layout()
savefig("q2_q3_label_distribution")
plt.show()

n_benign, n_attack = (df.is_attack == 0).sum(), (df.is_attack == 1).sum()
print(f"Takeaway: {n_attack / len(df):.1%} of flows are attacks. Imbalance ratio (benign : attack) = {n_benign / max(n_attack, 1):.1f} : 1")
rare = counts[(counts < 1000) & (counts.index.str.upper() != "BENIGN")]
print("Very rare attack classes (< 1000 flows):", {k: int(v) for k, v in rare.items()} if len(rare) else "none")

**What it means:** the vast majority of flows are benign, and the attack classes are very unequal in size. For the project this means:
* **Do not use accuracy** as the main metric.
* An **autoencoder trained on benign traffic only** is a natural fit: it learns "normal" and flags what does not look normal, so it does not need many attack examples.
* Very rare attacks (a few flows) will be nearly impossible to evaluate separately, so we evaluate mainly **attack vs benign**.

## Q4. Are there missing or infinite values?
## Q5. Are there duplicate rows?
## Q6. Are there useless columns (constant values) or strange values (negatives)?
**Why we ask:** neural networks (LSTM / GRU) **cannot accept `NaN` or `inf`**: one bad value can turn the whole loss into `NaN`. Duplicates inflate the counts and can leak between train and test. Constant columns carry no information.

**Where the infinite values come from:** `Flow Bytes/s = bytes / duration` and `Flow Packets/s = packets / duration`. When a flow has duration 0, the division gives `inf` (or `NaN`).

> **In this cell we only measure the problems. We do not delete anything yet.** The numbers go into the report.

In [ ]:
X = df[feat]
inf_mask = np.isinf(X)

quality = pd.DataFrame({
    "inf_values": inf_mask.sum(),
    "nan_values": X.isna().sum(),
    "negative_values": (X < 0).sum(),
    "unique_values": X.nunique(),
})

# Q4: missing / infinite
bad_rows = (inf_mask | X.isna()).any(axis=1)
print(f"Q4. Rows containing NaN or inf: {bad_rows.sum():,} ({bad_rows.mean():.3%})")
print("    Share of attacks inside those rows:", f"{df.loc[bad_rows, 'is_attack'].mean():.1%}" if bad_rows.any() else "n/a")
display(quality[(quality.inf_values > 0) | (quality.nan_values > 0)])

# Q5: duplicates (exact duplicates = every column identical, including Timestamp and IPs)
cols_for_dup = [c for c in df.columns if c != "source_file"]
exact_dups = df.duplicated(subset=cols_for_dup).sum()
feature_dups = df.duplicated(subset=feat + ["Label"]).sum()
print(f"\nQ5. Exact duplicate rows            : {exact_dups:,} ({exact_dups / len(df):.2%})")
print(f"    Same feature values (any time)  : {feature_dups:,} ({feature_dups / len(df):.2%})  <- information only, NOT removed")

# Q6: constant columns and negative values
const_cols = quality.index[quality.unique_values <= 1].tolist()
neg_cols = quality.index[quality.negative_values > 0].tolist()
print(f"\nQ6. Constant columns (no information): {const_cols}")
print(f"    Columns containing negative values : {len(neg_cols)}")
display(quality.loc[neg_cols, ["negative_values"]].sort_values("negative_values", ascending=False).head(10))

**What it means and what we decide:**
* **NaN / inf:** only a tiny fraction of rows is affected, so we **drop these rows** (imputing would invent traffic that never existed). We checked the attack share inside them to be sure we are not removing a whole attack class.
* **Duplicates:** we remove only **exact** duplicates (identical in every column, including the timestamp and IPs). We do **not** remove rows that merely have the same feature values at different times, because in a time series a repeated pattern (for example a DoS tool sending identical requests) is real behaviour.
* **Constant columns:** useless for learning, so we drop them.
* **Negative values:** some columns legitimately contain negative numbers (for example `Init_Win_bytes_forward = -1` means "no window was set"). Because of that we keep them, and we later use a **signed** log transform that supports negative values.

## 5. Cleaning
Order of operations: (1) replace `inf` with `NaN` and drop rows with `NaN`, (2) drop exact duplicates, (3) drop constant columns, (4) store the features as `float32` to halve the memory.

We also compare the label counts **before and after** to prove that no attack class disappeared.

In [ ]:
labels_before = df["Label"].value_counts()
steps = [("Raw data", len(df))]

df[feat] = df[feat].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=feat)
steps.append(("After dropping NaN / inf rows", len(df)))

df = df.drop_duplicates(subset=[c for c in df.columns if c != "source_file"])
steps.append(("After dropping exact duplicates", len(df)))

nunique = df[feat].nunique()
const_cols = nunique[nunique <= 1].index.tolist()
df = df.drop(columns=const_cols)
feat = [c for c in feat if c not in const_cols]

df[feat] = df[feat].astype("float32")
df = df.reset_index(drop=True)

summary = pd.DataFrame(steps, columns=["step", "rows"])
summary["removed"] = (-summary["rows"].diff()).fillna(0).astype(int)
display(summary)
print(f"Dropped {len(const_cols)} constant column(s): {const_cols}")
print(f"Features remaining: {len(feat)}")

comparison = pd.DataFrame({"before": labels_before, "after": df["Label"].value_counts()}).fillna(0).astype(int)
comparison["kept_%"] = (comparison["after"] / comparison["before"] * 100).round(1)
print("\nLabel counts before vs after cleaning:")
display(comparison)
lost = comparison[comparison["after"] == 0].index.tolist()
print("Classes completely removed by cleaning:", lost if lost else "none")

## 6. Fix the timestamps and sort in time order
**Why this step is critical:** our project is a **time-series** problem. A sliding window only makes sense if the flows are in the **correct chronological order**.

**A known quirk of this dataset:** the `Timestamp` is stored in a 12-hour style **without AM/PM**, so an afternoon flow at 15:30 appears as `3:30`. If we sorted these values directly, afternoon flows would appear *before* morning flows. The capture only ran from about 8:00 to 17:00, so any hour from 1 to 7 must be afternoon and we add 12 hours to it. We print the time range of every file **before and after** so you can verify the fix.

In [ ]:
assert "Timestamp" in df.columns, ("There is no Timestamp column. You probably downloaded MachineLearningCSV; "
                                   "use GeneratedLabelledFlows instead.")

# Day-first dates, mixed formats (some files have seconds, some do not)
try:
    ts = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce", format="mixed")
except (TypeError, ValueError):                       # older pandas versions
    ts = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")

print("Unparsable timestamps:", int(ts.isna().sum()))
before = ts.groupby(df["source_file"]).agg(["min", "max"])

FIX_PM = True
if FIX_PM:
    is_pm = ts.dt.hour < 8                             # capture runs ~08:00-17:00, so 1..7 means afternoon
    ts = ts + pd.to_timedelta(is_pm.astype(int) * 12, unit="h")
after = ts.groupby(df["source_file"]).agg(["min", "max"])

print("\nTime range per file BEFORE the AM/PM fix:")
display(before)
print("Time range per file AFTER the AM/PM fix (should be roughly 08:00 - 17:00 for each day):")
display(after)

df["Timestamp"] = ts
df = df.dropna(subset=["Timestamp"]).sort_values("Timestamp", kind="stable").reset_index(drop=True)
print(f"\nData now covers {df['Timestamp'].min()}  ->  {df['Timestamp'].max()}")

---
# Part B: Exploring behaviour
## Q7. How are the main features distributed?
**Why we ask:** network features such as duration, packet counts and bytes per second have **huge ranges** (from 0 to hundreds of millions). A few giant values (outliers) can dominate a neural network. We measure the **skewness** (how long the right tail is) and test a `log` transform.

## Q8. Do attacks look different from normal traffic?
**Why we ask:** if attacks looked exactly like benign traffic, no model could detect them. We also want to know **which features separate them best**, to explain what the model is probably using.

We use `signed_log(x) = sign(x) * log(1 + |x|)`. It compresses huge values like a normal log, but it also handles negative values and zero.

In [ ]:
def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))

# Work on a random sample so plotting stays fast
sample = df.sample(min(len(df), 300_000), random_state=SEED)
sample = sample.assign(**{"class": sample["is_attack"].map({0: "Benign", 1: "Attack"})})
Z = signed_log(sample[feat].astype("float64"))        # log-transformed features (sample only)

# ---- Q7: skewness of raw features
skew = sample[feat].skew().sort_values(ascending=False)
print("Q7. Most skewed raw features:")
display(skew.head(8).to_frame("skewness"))
print(f"Features with |skewness| > 2: {(skew.abs() > 2).sum()} out of {len(feat)}  -> a log transform is needed before scaling")

# ---- Q7 + Q8: histograms of key features, benign vs attack, after the log transform
preferred = ["Flow Duration", "Total Fwd Packets", "Total Backward Packets", "Flow Bytes/s",
             "Flow Packets/s", "Fwd Packet Length Mean", "Flow IAT Mean", "Packet Length Mean"]
key_feats = [c for c in preferred if c in feat][:8]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.ravel(), key_feats):
    plot_df = pd.DataFrame({"value": Z[col].values, "class": sample["class"].values})
    sns.histplot(data=plot_df, x="value", hue="class", palette=PALETTE, bins=50, stat="density",
                 common_norm=False, element="step", ax=ax, legend=(col == key_feats[0]))
    ax.set_title(col)
    ax.set_xlabel("signed_log(value)")
for ax in axes.ravel()[len(key_feats):]:
    ax.axis("off")
plt.suptitle("Q7 / Q8. Key features after log transform: benign vs attack", y=1.02, fontsize=14)
plt.tight_layout()
savefig("q7_q8_key_feature_distributions")
plt.show()

In [ ]:
# ---- Q8: median value of key features for each label (median is robust to outliers)
print("Q8. Median of key features per label:")
display(df.groupby("Label")[key_feats].median())

# ---- Q8: which features separate benign from attack the most?
# Standardised mean difference = |mean_attack - mean_benign| / pooled std (on log-transformed values)
a, b = Z[sample.is_attack == 1], Z[sample.is_attack == 0]
pooled = np.sqrt((a.var() + b.var()) / 2).replace(0, np.nan)
separation = ((a.mean() - b.mean()) / pooled).abs().sort_values(ascending=False).dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 1.6]})
separation.head(12)[::-1].plot(kind="barh", ax=axes[0], color="#55A868")
axes[0].set_title("Top 12 features that separate attack from benign")
axes[0].set_xlabel("Standardised mean difference (bigger = more different)")

top4 = separation.head(4).index.tolist()
long = (Z[top4].assign(**{"class": sample["class"].values})
        .melt(id_vars="class", var_name="feature", value_name="signed_log(value)"))
sns.boxplot(data=long, x="feature", y="signed_log(value)", hue="class", palette=PALETTE, ax=axes[1], fliersize=1)
axes[1].set_title("Distribution of the 4 most separating features")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
savefig("q8_feature_separation")
plt.show()

print("Takeaway: most separating features ->", ", ".join(separation.head(5).index))

**What it means:**
* Almost all features are **heavily right-skewed**, so before training we must apply the log transform and then scale. Without it, a few extreme flows would dominate the network.
* The benign and attack histograms **overlap but are clearly different** for several features, so attacks are detectable from flow statistics. Different attack types differ in different ways (for example DoS flows are very long with idle gaps, while a PortScan has tiny flows with few packets).
* The overlap also explains why **some false positives and false negatives are unavoidable**. We should say this in the limitations.

## Q9. When do attacks happen, and do they come in bursts?
**Why we ask:** this is the most important question for a **time-series** project. If attacks were scattered randomly, one flow at a time, a normal row-by-row classifier would be enough. If they come in **bursts**, then looking at a **window of consecutive flows** gives the model much more information.

We look at three things: (1) flows per 5 minutes on a timeline, (2) when each attack starts and ends, (3) how long consecutive runs of attack flows are.

In [ ]:
# (1) Flows per 5 minutes, benign vs attack
per_bin = (df.set_index("Timestamp").groupby([pd.Grouper(freq="5min"), "is_attack"]).size()
             .unstack(fill_value=0).reindex(columns=[0, 1], fill_value=0)
             .rename(columns={0: "Benign", 1: "Attack"}))

fig, ax = plt.subplots(figsize=(15, 4.5))
for name in ["Benign", "Attack"]:
    ax.plot(per_bin.index, per_bin[name].replace(0, np.nan), ".-", ms=3, lw=0.8, label=name, color=PALETTE[name])
ax.set_yscale("log")
ax.set_ylabel("Flows per 5 minutes (log)")
ax.set_title("Q9. Traffic over time: attacks appear in bursts, not randomly")
ax.legend()
plt.tight_layout()
savefig("q9_timeline")
plt.show()

# (2) When does each attack start and end?
timing = (df[df.is_attack == 1].groupby("Label")["Timestamp"]
            .agg(first_flow="min", last_flow="max", flows="size"))
timing["duration_min"] = ((timing["last_flow"] - timing["first_flow"]).dt.total_seconds() / 60).round(1)
print("When does each attack happen?")
display(timing.sort_values("first_flow"))

# (3) Temporal structure: how persistent is the class from one flow to the next?
y = df["is_attack"].to_numpy()
persistence = (y[1:] == y[:-1]).mean()
run_id = np.cumsum(np.r_[1, y[1:] != y[:-1]])
run_lengths = pd.Series(run_id[y == 1]).value_counts()          # length of each run of attack flows
top10_bins = per_bin["Attack"].nlargest(10).sum() / max(per_bin["Attack"].sum(), 1)

print(f"\nProbability that the next flow has the SAME class as the current one: {persistence:.1%}")
print(f"Attack runs: {len(run_lengths):,} runs | median length {run_lengths.median():.0f} flows | longest {run_lengths.max():,} flows")
print(f"The 10 busiest 5-minute bins contain {top10_bins:.1%} of all attack flows")

**What it means:** the next flow almost always has the **same class** as the current one, and attacks form **long continuous runs** inside short time periods. This is the evidence for the question *"why a sequence model and not a row-by-row classifier?"*:
* One isolated flow can look harmless, but **20 consecutive flows with the same suspicious pattern** (a flood, a scan, a brute-force loop) are much easier to recognise.
* The **order and timing** of flows carry information that is lost if we treat rows as independent.

**Also note for the data split:** attacks are concentrated in specific days, so we must split by **time/day**, never randomly.

## Q10. Does the same source IP send many flows in a row?
**Why we ask:** a single machine that sends thousands of flows in a short time is typical of a DoS, a brute-force tool or a scan. This is **sequential behaviour** that a window of flows can capture and a single row cannot.

In [ ]:
if "Source IP" not in df.columns:
    print("No 'Source IP' column in this file version. Skipping Q10.")
else:
    by_ip = (df.groupby("Source IP").agg(flows=("is_attack", "size"), attack_flows=("is_attack", "sum"))
               .assign(benign_flows=lambda t: t.flows - t.attack_flows)
               .sort_values("flows", ascending=False))
    top10_share = by_ip.head(10).flows.sum() / len(df)

    print(f"Unique source IPs: {len(by_ip):,}")
    print(f"Flows per IP -> median {by_ip.flows.median():.0f}, mean {by_ip.flows.mean():.0f}, max {by_ip.flows.max():,}")
    print(f"The top 10 IPs send {top10_share:.1%} of all flows")

    same_ip_next = (df["Source IP"].to_numpy()[1:] == df["Source IP"].to_numpy()[:-1]).mean()
    print(f"Probability that the next flow in time comes from the same source IP: {same_ip_next:.1%}")

    attackers = by_ip[by_ip.attack_flows > 0].head(8)
    print("\nSource IPs that send the most attack flows:")
    display(attackers)

    top = by_ip.head(10)
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.bar(top.index, top.benign_flows, label="Benign", color=PALETTE["Benign"])
    ax.bar(top.index, top.attack_flows, bottom=top.benign_flows, label="Attack", color=PALETTE["Attack"])
    ax.set_title("Q10. Top 10 source IPs by number of flows")
    ax.set_ylabel("Flows")
    ax.tick_params(axis="x", rotation=40)
    ax.legend()
    plt.tight_layout()
    savefig("q10_top_source_ips")
    plt.show()

**What it means:** a small number of IPs produce a very large share of the flows, and the attack traffic comes from a **few attacker machines** that send huge numbers of flows in a row. Be careful: this is also a **limitation**. In this lab the attackers have fixed IP addresses, so a model could "cheat" by memorising them. That is exactly why we **do not use IPs and ports as model features**.

---
# Part C: Correlation analysis
## C1. Which features are strongly correlated with each other?
**Why we ask:** many CICFlowMeter features are computed from each other (for example `Fwd Packet Length Mean` and `Avg Fwd Segment Size` are almost the same number). Keeping both:
* adds **no new information** (redundancy),
* makes the LSTM **bigger and slower** (more inputs for every time step),
* can make training less stable.

## C2. Which features are most correlated with being an attack?
**Why we ask:** it gives a first idea of which features matter. Pearson correlation measures only **linear** relationships, so it is a rough guide and not a proof.

We calculate correlations on the **log-transformed** values, because raw values are so skewed that a few outliers would distort the result.

In [ ]:
Zc = signed_log(sample[feat].astype("float64"))
corr = Zc.corr().fillna(0)

# ---- C1: heatmap (lower triangle only, so each pair appears once)
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
            xticklabels=False, yticklabels=False, cbar_kws={"shrink": 0.6})
plt.title("C1. Feature-to-feature correlation (log-transformed)")
savefig("c1_correlation_heatmap")
plt.show()

THRESHOLD = 0.95
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().rename("correlation").reset_index()
pairs.columns = ["feature_1", "feature_2", "correlation"]
high = pairs[pairs.correlation.abs() > THRESHOLD].sort_values("correlation", key=abs, ascending=False)
print(f"Feature pairs with |correlation| > {THRESHOLD}: {len(high)}  (> 0.90: {(pairs.correlation.abs() > 0.90).sum()})")
display(high.head(12))

# ---- C2: correlation of every feature with is_attack
target_corr = Zc.corrwith(sample["is_attack"]).dropna().sort_values(key=abs, ascending=False)
plt.figure(figsize=(8, 6))
target_corr.head(15)[::-1].plot(kind="barh", color=np.where(target_corr.head(15)[::-1] > 0, PALETTE["Attack"], PALETTE["Benign"]))
plt.title("C2. Top 15 features correlated with is_attack")
plt.xlabel("Pearson correlation")
plt.tight_layout()
savefig("c2_target_correlation")
plt.show()

## C3. Which features do we keep?
**Rule:** go through the features in their original order. Keep a feature **only if** its correlation with every feature already kept is at most `0.95`; otherwise drop it and record **which kept feature it duplicates**.

**Why this rule:** it is simple, reproducible and **does not use the label**, so it cannot leak information about attacks into the feature selection. (A smarter choice, such as keeping the feature most related to the label, is possible, but then the selection should be done on the training split only.)

In [ ]:
kept = []
for c in corr.columns:
    if all(abs(corr.loc[c, k]) <= THRESHOLD for k in kept):
        kept.append(c)
dropped = [c for c in corr.columns if c not in kept]

drop_report = pd.DataFrame(
    [(c, corr.loc[c, kept].abs().idxmax(), round(float(corr.loc[c, kept].abs().max()), 3)) for c in dropped],
    columns=["dropped_feature", "duplicates_kept_feature", "abs_correlation"])

print(f"Features before: {len(feat)} | kept: {len(kept)} | dropped as redundant: {len(dropped)}")
display(drop_report.head(15))

final_features = kept

**What it means:** we remove a good part of the features without losing information, so each time step fed to the LSTM is smaller and training is faster. The `drop_report` table is the **justification** to put in the report: every removed feature is almost a copy of a feature we kept.

---
# Part D: Save the results for the team
We save:
* `clean.parquet`: the cleaned, time-ordered data (identifiers + `Label` + `is_attack` + features), used by whoever builds the windows and the models.
* `final_features.json`: the list of features to use as model input.
* `dropped_features.csv`, `label_counts.csv`, `eda_summary.json`: supporting tables for the report and slides.

In [ ]:
keep_columns = [c for c in ["Timestamp", "Source IP", "Destination IP", "Destination Port", "source_file"] if c in df.columns]
clean = df[keep_columns + ["Label", "is_attack"] + final_features]

try:
    clean.to_parquet(OUT_DIR / "clean.parquet", index=False)
    saved_as = "clean.parquet"
except Exception as e:                                   # pyarrow missing -> fall back to compressed CSV
    print("Parquet not available (", type(e).__name__, ") -> saving clean.csv.gz instead. Tip: pip install pyarrow")
    clean.to_csv(OUT_DIR / "clean.csv.gz", index=False, compression="gzip")
    saved_as = "clean.csv.gz"

(OUT_DIR / "final_features.json").write_text(json.dumps(final_features, indent=2))
drop_report.to_csv(OUT_DIR / "dropped_features.csv", index=False)
label_table.to_csv(OUT_DIR / "label_counts.csv")

summary_stats = {
    "rows_raw": int(summary["rows"].iloc[0]),
    "rows_clean": int(len(df)),
    "attack_percent": round(float(df.is_attack.mean() * 100), 2),
    "n_labels": int(df["Label"].nunique()),
    "features_after_cleaning": len(feat),
    "features_final": len(final_features),
    "time_start": str(df["Timestamp"].min()),
    "time_end": str(df["Timestamp"].max()),
    "next_flow_same_class_percent": round(float(persistence * 100), 2),
}
(OUT_DIR / "eda_summary.json").write_text(json.dumps(summary_stats, indent=2))

print(f"Saved to {OUT_DIR.resolve()}: {saved_as}, final_features.json, dropped_features.csv, label_counts.csv, eda_summary.json")
print(json.dumps(summary_stats, indent=2))

---
# Part E: Summary of findings and decisions
*(The numbers above are printed from your own run, so copy the exact values into the report. The statements below describe what is typically found in CICIDS2017; confirm each one against your outputs.)*

| # | Finding | Decision / consequence |
|---|---|---|
| 1 | About 2.8 million flows, about 78 numeric features, 15 labels (14 attack types) when all files are used | Use `SAMPLE_FRAC` or a subset of days if memory is limited |
| 2 | Roughly 80% benign, 20% attack; some attacks have only a handful of flows | Do **not** use accuracy. Use precision, recall, F1, ROC-AUC. Evaluate mainly attack vs benign |
| 3 | `inf` / `NaN` values come from flows with zero duration; they affect a tiny share of rows | Drop those rows |
| 4 | Exact duplicate rows exist | Remove exact duplicates only (keep repeated patterns at different times) |
| 5 | A few constant columns | Drop them |
| 6 | Timestamps have no AM/PM | Fix the hour, then sort by time |
| 7 | Features are extremely right-skewed | Apply `signed_log` before scaling |
| 8 | Several features clearly separate attack from benign | Attacks are detectable from flow statistics, but overlap means some errors are unavoidable |
| 9 | The next flow almost always has the same class; attacks come in long bursts inside specific days | Justifies **sliding windows / sequence models**; the split must be by time or day, never random |
| 10 | A few source IPs send most flows | IPs and ports are **excluded** from the features to avoid shortcut learning |
| 11 | Many features are almost copies of each other (|corr| > 0.95) | Keep one per group (`final_features.json`) |

## Hand-off to Person 2 (modelling)
1. Load `clean.parquet` and `final_features.json`.
2. **Split by time or day** (for example train on benign data from the earlier period, validate and test on later periods that contain attacks).
3. Apply `signed_log`, then **fit the scaler on the training benign data only**, to avoid leakage.
4. Build sliding windows (for example window = 20 flows) and label a window as attack if **any** flow in it is an attack.

## Limitations to mention in the presentation
* Lab-generated 2017 traffic, so results may not transfer directly to a real network.
* Strong class imbalance and a few very rare attack classes.
* Possible label noise in the original CSV files.
* A model trained on these attacks may miss **zero-day** attacks and may raise **false alarms** on unusual but harmless traffic.